In [1]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')


In [2]:
train = pd.read_csv("/kaggle/input/titanic/train.csv")
test  = pd.read_csv("/kaggle/input/titanic/test.csv")


In [3]:
train.info()
train.isnull().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [4]:
datasets = [train, test]


In [5]:
for df in datasets:
    df['Age'].fillna(df['Age'].median(), inplace=True)
    df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

for df in datasets:
    if 'Cabin' in df.columns:
        df.drop('Cabin', axis=1, inplace=True)


In [6]:
for df in datasets:
    df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

for df in datasets:
    df['Title'] = df['Title'].replace(
        ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona'],
        'Rare'
    )
    df['Title'] = df['Title'].replace({'Mlle':'Miss','Ms':'Miss','Mme':'Mrs'})


In [7]:
title_map = {'Mr':1, 'Miss':2, 'Mrs':3, 'Master':4, 'Rare':5}
for df in datasets:
    df['Title'] = df['Title'].map(title_map)
    df['Title'].fillna(0, inplace=True)


In [8]:
for df in datasets:
    df['Sex'] = df['Sex'].map({'male':0, 'female':1})


In [9]:
for df in datasets:
    df['Embarked'] = df['Embarked'].map({'C':0, 'Q':1, 'S':2})
    df['Embarked'].fillna(0, inplace=True)


In [10]:
for df in datasets:
    df.loc[df['Age'] <= 16, 'Age'] = 0
    df.loc[(df['Age'] > 16) & (df['Age'] <= 32), 'Age'] = 1
    df.loc[(df['Age'] > 32) & (df['Age'] <= 48), 'Age'] = 2
    df.loc[(df['Age'] > 48) & (df['Age'] <= 64), 'Age'] = 3
    df.loc[df['Age'] > 64, 'Age'] = 4


In [11]:
for df in datasets:
    df['Fare'] = np.log1p(df['Fare'])


In [12]:
for df in datasets:
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)


In [13]:
test_ids = test['PassengerId']

for df in datasets:
    df.drop(['Name', 'Ticket', 'PassengerId'], axis=1, inplace=True)


In [14]:
X = train.drop('Survived', axis=1)
y = train['Survived']


In [15]:
X.dtypes
test.dtypes


Pclass          int64
Sex             int64
Age           float64
SibSp           int64
Parch           int64
Fare          float64
Embarked        int64
Title           int64
FamilySize      int64
IsAlone         int64
dtype: object

In [16]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)

model.fit(X, y)


RandomForestClassifier(max_depth=6, min_samples_leaf=5, min_samples_split=10,
                       n_estimators=300, random_state=42)

In [17]:

test['Fare'].fillna(test['Fare'].median(), inplace=True)



In [18]:
prediction = model.predict(test)


In [19]:
prediction[:10]

array([0, 0, 0, 0, 1, 0, 1, 0, 1, 0])

In [20]:
submission = pd.DataFrame({
    "PassengerId": test_ids,
    "Survived": prediction
})

submission.to_csv("submission.csv", index=False)
print("✅ submission.csv saved successfully")


✅ submission.csv saved successfully
